In [ ]:
# wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
# sudo dpkg -i cuda-keyring_1.1-1_all.deb
# sudo apt-get update
# sudo apt-get -y install cuda-toolkit-13-3
# # sudo apt-get install -y nvidia-open
# # sudo apt-get install -y cuda-drivers

# uv add spacy spacy-lookups-data 'spacy[cuda11x]' 'cupy-cuda11x[ctk]'
# # uv run spacy download ru_core_news_sm
# uv run spacy download ru_core_news_md

# uv run python -c "import spacy; print(spacy.require_gpu())"


# uv add --dev ipykernel
# uv run ipython kernel install --user --env VIRTUAL_ENV $(pwd)/.venv --name=uv-kernel
# jupyter kernelspec list


In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
import re

import numpy as np
import pandas as pd
import polars as pl
import polars.lazyframe as plf

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)


In [ ]:
dataset_fps = list(RAW_DIR.glob('*.parquet'))

df = pl.concat([pl.scan_parquet(fp) for fp in dataset_fps[:2]])
df.head().count().collect(), df.head().collect()


In [ ]:
import re
import pymorphy3

def clean_and_normalize_russian_text_advanced(text: str) -> list:
    # 1. Initialize the Morphological Analyzer
    morph = pymorphy3.MorphAnalyzer()
    
    # 2. Convert to lowercase
    text_lowercase = text.lower()
    
    # 3. Advanced Tokenization via Regex
    # Pattern breakdown:
    # [а-яё]+       -> Matches one or more Russian letters
    # (?:-[а-яё]+)* -> Matches an optional internal hyphen followed by more Russian letters
    #                  The '*' allows multiple hyphens (e.g., "рок-н-ролл")
    word_pattern = r'[а-яё]+(?:-[а-яё]+)*'
    words = re.findall(word_pattern, text_lowercase)
    
    cleaned_words = []
    
    # 4. Filter by Part of Speech and Normalize
    for word in words:
        parsed = morph.parse(word)[0]
        pos = parsed.tag.POS
        
        # Keep only Nouns and Verbs (including infinitives)
        if pos in ['NOUN', 'VERB', 'INFN']:
            cleaned_words.append(parsed.normal_form)
            
    return cleaned_words

# --- Comprehensive Edge-Case Testing ---
dirty_text = """
В 2026 году бизнес-план компании "Окна-Маркет" по-прежнему лежал на столе.
Кто-то сказал: "Это же рок-н-ролл!" — и начал быстро-быстро писать код.
Из-за форс-мажора пришлось всё переделывать.
"""

result = clean_and_normalize_russian_text_advanced(dirty_text)
print(result)


In [ ]:
# import re
# import unicodedata
# import spacy

# spacy.require_gpu()

# nlp = spacy.load("ru_core_news_md", disable=["parser", "ner"])

# # Regex pattern to match individual Cyrillic words
# RU_WORD_RE = re.compile(r'^[а-яёА-ЯЁ]+$')


# def extract_russian_nouns_and_verbs(text: str) -> list:
#     if not text:
#         return []

#     # 3. Unicode Normalization
#     # NFKD decomposes combined characters (e.g., 'а́' becomes 'а' + '\u0301')
#     decomposed = unicodedata.normalize('NFKD', text)
#     # Strip away the Combining Acute Accent (\u0301) which represents the stress mark
#     stripped = "".join([c for c in decomposed if ord(c) != 0x0301])
#     # Recompose back into standard standard format (NFC)
#     cleaned_text = unicodedata.normalize('NFC', stripped)

#     # 4. Process text through the GPU-accelerated pipeline
#     doc = nlp(cleaned_text)
    
#     tokens = list(doc)
#     results = []
#     i = 0
#     n = len(tokens)

#     # 5. Extract and reconstruct compound words
#     while i < n:
#         token = tokens[i]

#         # Check if the token is written in the Russian alphabet
#         if RU_WORD_RE.match(token.text):
#             current_group = [token]

#             # Look ahead to catch hyphenated compound words with no spaces (e.g., "диваны-кровати")
#             while (i + 2 < n and 
#                    tokens[i].whitespace_ == "" and 
#                    tokens[i+1].text == "-" and 
#                    tokens[i+1].whitespace_ == "" and 
#                    RU_WORD_RE.match(tokens[i+2].text)):
#                 current_group.extend([tokens[i+1], tokens[i+2]])
#                 i += 2  # Advance past the hyphen and the next sub-word

#             # Isolate the linguistic components (ignoring the hyphens)
#             word_tokens = [t for t in current_group if t.text != "-"]

#             # Filter for Nouns and Verbs
#             # SpaCy POS tags: NOUN (nouns), PROPN (proper nouns), VERB (verbs), AUX (auxiliary verbs)
#             is_target_pos = any(t.pos_ in {'NOUN', 'PROPN', 'VERB', 'AUX'} for t in word_tokens)

#             if is_target_pos:
#                 # Reconstruct the normal form (lemma) of the compound word
#                 lemma = "".join([t.lemma_ for t in current_group])
#                 results.append(lemma)

#             i += 1
#         else:
#             i += 1

#     return results

# extract_russian_nouns_and_verbs(dirty_text)


In [ ]:
from concurrent.futures import ProcessPoolExecutor

from tqdm import tqdm


pl.Config.set_streaming_chunk_size(500)
executor = ProcessPoolExecutor(max_workers=10)


def process_batch(df: pl.DataFrame) -> pl.DataFrame:
    # print('batch size:', df.shape[0])

    # fulltext = " ".join(df['text'].to_list())
    # words = clean_and_normalize_russian_text_advanced(fulltext)
    # words = extract_russian_nouns_and_verbs(fulltext)
    # words = fulltext.split()
    
    words = executor.map(clean_and_normalize_russian_text_advanced, df['text'].to_list())
    # words = executor.map(extract_russian_nouns_and_verbs, df['text'].to_list())
    # words = [line.split() for line in df['text'].to_list()]

    pbar.update(df.shape[0])
    # print(words)
    return pl.DataFrame({'clean_text': words})


for DATA_SKIP in range(80_000, 200_000, 20_000):
    # DATA_SKIP = 50
    DATA_TAKE = 20

    print(f'Processing rows {DATA_SKIP} to {DATA_SKIP + DATA_TAKE}')
    pbar = tqdm(total=DATA_TAKE, desc="Processing rows")

    res = df.slice(DATA_SKIP, DATA_TAKE).map_batches(process_batch, streamable=True, schema=pl.Schema({'clean_text': pl.List(pl.String)}))\
        .select('clean_text').explode('clean_text', empty_as_null=True, keep_nulls=False).group_by('clean_text').agg(pl.len()).sort('len', descending=True)
    res.sink_csv(DATA_DIR / 'processed' / f'wiki_words_frequency_{DATA_SKIP}_{DATA_SKIP+DATA_TAKE}.csv', engine='streaming')
    # print(res.collect().head())

    pbar.close()


executor.shutdown()
print('Done!')


In [ ]:
# res.collect().to_pandas()


In [ ]:
break

In [ ]:
from tqdm import tqdm

def track_progress(pbar, func):
    def wrapper(*args, **kwargs):
        pbar.update(1)
        return func(*args, **kwargs)
    return wrapper

DATA_SKIP = 5000
DATA_TAKE = 50
pbar = tqdm(total=DATA_TAKE, desc="Processing rows")


res = df.slice(DATA_SKIP, DATA_TAKE).with_columns(
    pl.col('text').map_elements(track_progress(pbar, extract_russian_nouns_and_verbs), return_dtype=pl.List(pl.String)).alias('clean_text')
).select(pl.col('clean_text')).explode('clean_text', empty_as_null=True, keep_nulls=False).group_by('clean_text').agg(pl.len()).sort('len', descending=True)
res.sink_csv(DATA_DIR / 'processed' / f'wiki_words_frequency_{DATA_SKIP}_{DATA_SKIP+DATA_TAKE}.csv', engine='streaming')


pbar.close()
